
# ỨNG DỤNG CLUSTERING ĐỂ PHÁT HIỆN ẢNH TRÙNG

1. Yêu cầu chung: Dùng kỹ thuật clustering để tạo công cụ hỗ trợ phát hiện các ảnh trùng nhau

2. Yêu cầu cụ thể:
  - Input: Danh sách các ảnh được lưu trong tập tin, ví dụ CarDataset-Splits-1-Train.csv (xem mô tả https://colab.research.google.com/drive/1gf0GzvW0tHddKtuvMUNIvglUT-J6oW6S?usp=sharing)
  - Output: Danh sách các clusters và hiển thị các ảnh trong cluster

3. Hướng dẫn:
  - Bước 1:
    - Mỗi ảnh cần thực hiện bước rút trích đặc trưng (feature extraction), biểu diễn dưới dạng một vector đặc trưng d chiều (d-dimension).
    - Có nhiều công cụ hỗ trợ bước rút trích đặc trưng, trong bài tập này, chúng ta sẽ chọn một công cụ sao cho tốc độ xử lý nhanh nhưng kết quả tốt. Các mô hình MobileNet (https://keras.io/api/applications/mobilenet/) có thể được dùng vì đáp ứng các tiêu chí này.
  - Bước 2:
    - Chọn một thuật toán clustering - ví dụ K-Means (số lượgn clusters K=5)
    - Ghi kết quả clustering ra tập tin - thay CategoryID bằng ClusterID
  - Bước 3:
    - Hiển thị kết quả clustering - kế thừa kết quả của bài tập Hiển thị dữ liệu https://colab.research.google.com/drive/1rHbKlJd7O9E49SsJlHnZNKcyTbwXT_Ls?usp=sharing
    - Từ kết quả hiển thị, nếu các ảnh nhìn trùng nhau, nhưng tên tập tin khác nhau thì có thể đưa vào danh sách hậu kiểm.

In [ ]:
!pip install tensorflow

In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import os

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
csv_path = '/content/drive/MyDrive/CS114/Competition_1/data/CarDataset-Splits-1-Train.csv'
data = pd.read_csv(csv_path)
data.columns = ['ImageFullPath', 'CategoryID']
print(data.head())

                             ImageFullPath  CategoryID
0  Others/22520394-22520395.Others.130.jpg           0
1  Others/22520394-22520395.Others.131.jpg           0
2  Others/22520394-22520395.Others.133.jpg           0
3  Others/22520394-22520395.Others.134.jpg           0
4  Others/22520394-22520395.Others.135.jpg           0


In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH = 224

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

BATCH_SIZE = 48

#Tải hình và tiền xử lý
def load_and_preprocess_image(img_path):
    try:
        img = tf.io.read_file(img_path) # Đọc hình
        img = tf.image.decode_jpeg(img, channels=3) # Thêm channel màu RGB
        img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH]) # Resize về kích thước định sẵn
        img = preprocess_input(img)  # Chuẩn hóa ảnh (đưa dữ liệu trong vector ảnh về  -1 1)
        return img
    except Exception as e:
        print(f"Error loading image {img_path}: {e}")
        return None

# trích xuất đặc trưng từ DataFrame
def extract_features_and_labels(df, model):
    features = []
    labels = []
    ImgPath = []

    paths = [f'/content/drive/MyDrive/Public/{path}' for path in df["ImageFullPath"]]
    labels_list = df["CategoryID"].values
    # Tổng số batch
    total_batches = (len(paths) + BATCH_SIZE - 1) // BATCH_SIZE
    # Xử lý từng batch
    for batch_idx, i in enumerate(range(0, len(paths), BATCH_SIZE)):
        batch_paths = paths[i:i + BATCH_SIZE]
        batch_labels = labels_list[i:i + BATCH_SIZE]
        # Hiển thị thông tin batch
        print(f"Batch thứ {batch_idx + 1}/{total_batches}...")
        # Tải và xử lý batch ảnh
        batch_images = []
        for img_path in batch_paths:
            img = load_and_preprocess_image(img_path)
            if img is not None:
                batch_images.append(img)
                ImgPath.append(img_path)
        if len(batch_images) > 0:
            batch_images = tf.stack(batch_images)
            batch_features = model(batch_images, training=False)  # Trích xuất đặc trưng
            features.append(batch_features)
            labels.extend(batch_labels[:len(batch_images)])  # Lưu nhãn tương ứng

    if len(features) == 0:
        raise ValueError("Thất bại, không mở được file nào !")
    # Ghép các batch lại
    features = tf.concat(features, axis=0).numpy()
    labels = np.array(labels)
    ImgPath = np.array(ImgPath)

    return features, labels, ImgPath


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D

base_model = MobileNetV2(weights='imagenet', include_top=False)
model = Sequential([
    base_model,
    GlobalAveragePooling2D()
])

<ipython-input-9-7c02879d7813>:4: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False)


In [ ]:
try:
    features, labels, ImgPath = extract_features_and_labels(data, model)
except ValueError as e:
    print(f"Error during processing: {e}")

In [ ]:
# Define file paths
train_csv_path = "/content/drive/MyDrive/CS114/Competition_1/data/CarDataset-Splits-1-Train.csv"
duplicate_csv_path = "/content/drive/MyDrive/CS114/Competition_1/duplicate/duplicate_cluster8.csv"
output_csv_path = "/content/drive/MyDrive/CS114/Competition_1/CarDataset-Cluster.csv"

# Load the datasets
train_data = pd.read_csv(train_csv_path, names=['ImageFullPath', 'CategoryID'], header=0)
duplicate_data = pd.read_csv(duplicate_csv_path, names=['Cluster','ImageFullPath'], header=0)  # Added 'ClusterID'

# Filter rows from duplicate_data that exist in train_data
filtered_data = duplicate_data[duplicate_data['ImageFullPath'].isin(train_data['ImageFullPath'])]

# Save the filtered data, including 'ClusterID'
filtered_data.to_csv(output_csv_path, index=False)

print("Filtered dataset saved to:", output_csv_path)

Filtered dataset saved to: /content/drive/MyDrive/CS114/Competition_1/CarDataset-Cluster.csv


In [ ]:
data.head()

,0,1,2,3,4,5,6,7,8,9,...,1272,1273,1274,1275,1276,1277,1278,1279,label,ImgFullPath
0,1.530491,0.242274,0.000000,0.588050,0.086045,1.049449,0.006599,0.000000,0.000000,0.00000,...,0.000000,0.356413,0.029506,0.000000,0.164446,0.356747,0.804020,0.000000,0,Others/21522373-21522499.LandRover.1.jpg
1,1.162420,0.018673,0.027568,0.382832,0.048198,0.489705,0.000000,0.038598,0.115838,0.00000,...,0.013677,0.038112,0.045018,0.229037,0.051643,0.572668,0.916746,0.000000,0,Others/21522373-21522499.LandRover.10.jpg
2,1.094753,0.222335,0.000000,0.004441,0.360475,0.681041,0.000000,0.000000,0.000000,0.33261,...,0.000000,0.053114,0.036948,0.068911,0.000000,0.520736,1.038620,0.350059,0,Others/21522373-21522499.LandRover.11.jpg
3,1.036768,0.000000,0.099566,0.089183,0.027783,0.325496,0.000000,0.226533,0.000000,0.00000,...,0.000000,0.021895,0.387131,0.000000,0.000000,0.579457,0.478367,0.000000,0,Others/21522373-21522499.LandRover.12.jpg
4,0.369385,0.153411,0.015502,0.475488,0.264140,0.369472,0.003432,0.254214,0.000000,0.00000,...,0.000000,0.116366,0.000000,0.037027,0.013793,0.363459,0.054153,0.000000,0,Others/21522373-21522499.LandRover.13.jpg


In [6]:
import pandas as pd

split = 4
path = '/content/drive/MyDrive/CS114/Competition_1/data/extracted_feature.csv'
train_csv_path = f"/content/drive/MyDrive/CS114/Competition_1/data/CarDataset-Splits-{split}-Train.csv"
output_csv_path = "features.csv"
data = pd.read_csv(path)
train_data = pd.read_csv(train_csv_path, names=['ImageFullPath', 'CategoryID'], header=0)
filtered_data = data[data['ImgFullPath'].isin(train_data['ImageFullPath'])]
filtered_data.to_csv(output_csv_path, index=False)
print("Filtered dataset saved to:", output_csv_path)


Filtered dataset saved to: features.csv


In [ ]:
# k = 5  # Số lượng clusters
# kmeans = KMeans(n_clusters=k, random_state=42)
# cluster_labels = kmeans.fit_predict(features)

# # Ghi kết quả clustering ra tập tin
# output_df = pd.DataFrame({'image_dir': ImgPath, 'ClusterID': cluster_labels})
# output_csv_path = '/content/drive/MyDrive/CS114.P11/Train/CarDataset-Clusters.csv'
# output_df.to_csv(output_csv_path, index=False)
# print(f"Clustering results saved to {output_csv_path}")

Clustering results saved to /content/drive/MyDrive/CS114.P11/Train/CarDataset-Clusters.csv


In [21]:
# Lọc ảnh trùng sử dụng DBSCAN
from sklearn.cluster import DBSCAN
import numpy as np
import pandas as pd


feature_csv = pd.read_csv(output_csv_path)
features = feature_csv.drop(columns=["label", "ImgFullPath"])

db = DBSCAN(eps = 8,min_samples=2,metric="euclidean")
dup = db.fit_predict(features)

feature_csv['Cluster'] = dup
duplicate_clusters = feature_csv[feature_csv['Cluster'] >= 0]
result_df = duplicate_clusters[['Cluster', 'ImgFullPath']].sort_values(by='Cluster')
out_path = '/content/drive/MyDrive/CS114/Competition_1/CarDataset-Clusters.csv'
result_df.to_csv(out_path, index=False)
print(f"Clustering results saved to {out_path}")



Clustering results saved to /content/drive/MyDrive/CS114/Competition_1/CarDataset-Clusters.csv


In [ ]:
cluster_path = "/content/drive/MyDrive/CS114/Competition_1/CarDataset-Clusters.csv"
result_df = pd.read_csv(cluster_path)

def display_cluster_images(cluster_id, num_images=5):
    cluster_groups = result_df.groupby('Cluster')
    cluster_data = cluster_groups.get_group(cluster_id)  # Get data for the specific cluster
    image_paths = cluster_data['ImgFullPath'].tolist()  # Extract image paths

    plt.figure(figsize=(20, 10))

    for i, img_path in enumerate(image_paths[:num_images]):
        img_path = f'/content/drive/MyDrive/Public/{img_path}'
        img = Image.open(img_path)
        plt.subplot(1, num_images, i + 1)
        plt.imshow(img)
        plt.axis('off')

    plt.show()

for cluster_id, group in result_df.groupby('Cluster'):
    print(f"Cluster {cluster_id}:")
    print(group['ImgFullPath'].tolist())
    display_cluster_images(cluster_id)
    print("\n")

Output hidden; open in https://colab.research.google.com to view.